# Recurrent Neural Networks and LSTMs

This practice notebook demonstrates:

- One-step forecasting with an LSTM
- Inspecting gradients in a vanilla RNN
- The role of LSTM gates
- A simple memory-flow example
- Unrolling a recurrent model across time

The examples are intentionally small so the mechanics are easy to inspect.

## 1. LSTM for short-term energy-use forecasting

In [ ]:

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(25)

# A small illustrative sequence of daily energy readings.
readings = np.array(
    [12.5, 13.1, 12.8, 14.0, 14.6, 14.2, 15.0, 15.7, 15.1, 16.0, 16.4, 16.1],
    dtype=np.float32
)

def make_windows(values, lookback=4):
    inputs, labels = [], []
    for start in range(len(values) - lookback):
        inputs.append(values[start:start + lookback])
        labels.append(values[start + lookback])
    return np.asarray(inputs), np.asarray(labels)

lookback = 4
features, targets = make_windows(readings, lookback)
features = features[..., np.newaxis]

forecast_net = keras.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.LSTM(24),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

forecast_net.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss="mse"
)

forecast_net.fit(
    features,
    targets,
    epochs=35,
    batch_size=2,
    verbose=0
)

next_estimate = forecast_net.predict(features[-1: ], verbose=0)[0, 0]
print(f"Training examples: {len(features)}")
print(f"Estimated next reading: {next_estimate:.2f}")


## 2. Inspecting gradient magnitude in a vanilla RNN

In [ ]:

import torch
from torch import nn

torch.manual_seed(25)

class SequenceRegressor(nn.Module):
    def __init__(self, hidden_units=12):
        super().__init__()
        self.recurrent = nn.RNN(
            input_size=1,
            hidden_size=hidden_units,
            batch_first=True,
            nonlinearity="tanh"
        )
        self.output_layer = nn.Linear(hidden_units, 1)

    def forward(self, sequence):
        hidden_sequence, _ = self.recurrent(sequence)
        return self.output_layer(hidden_sequence)

rnn_model = SequenceRegressor()
loss_function = nn.MSELoss()
trainer = torch.optim.SGD(rnn_model.parameters(), lr=0.02)

# An alternating signal repeated over many time steps.
signal = ([0.0, 1.0, 0.0, 1.0, 1.0, 0.0] * 6)
desired = ([1.0, 0.0, 1.0, 0.0, 0.0, 1.0] * 6)

x = torch.tensor(signal, dtype=torch.float32).view(1, -1, 1)
y = torch.tensor(desired, dtype=torch.float32).view(1, -1, 1)

for round_number in range(4):
    trainer.zero_grad()
    prediction = rnn_model(x)
    error = loss_function(prediction, y)
    error.backward()

    hidden_gradient = rnn_model.recurrent.weight_hh_l0.grad
    print(
        f"Round {round_number + 1}: "
        f"loss={error.item():.5f}, "
        f"mean hidden-weight gradient={hidden_gradient.abs().mean().item():.7f}"
    )
    trainer.step()


## 3. LSTM gate intuition

At each time step, an LSTM uses three main gates:

- **Forget gate:** controls how much of the previous cell state remains.
- **Input gate:** controls how much new information is written.
- **Output gate:** controls how much of the cell state is exposed as the hidden state.

A simplified cell-state equation is:

\[
C_t = f_t \odot C_{t-1} + i_t \odot \widetilde{C}_t
\]

Here, \(\odot\) represents element-wise multiplication.

In [ ]:

from IPython.display import display, Markdown

display(Markdown(r'''
```text
Previous cell state C(t-1)
          |
          v
     [ Forget gate ] ----                          +----> New cell state C(t)
New input X(t) ----------/
          |
          v
      [ Input gate ]

New cell state C(t)
          |
          v
     [ Output gate ]
          |
          v
     Hidden state H(t)
```
'''))


## 4. Manual memory example: a user's travel searches

Assume a user searches for:

`["beach", "museum", "beach", "hotel"]`

For illustration, use binary gate values where `0` means block and `1` means pass.

### Step 1 — `"beach"`

- Forget gate = `0`: there is no earlier preference to retain.
- Input gate = `1`: store the beach preference.
- Cell state = `["beach"]`.
- Output gate = `1`: expose the current preference.

### Step 2 — `"museum"`

- Forget gate = `1`: retain the earlier beach preference.
- Input gate = `1`: add the museum interest.
- Cell state = `["beach", "museum"]`.
- Output gate = `1`: expose the combined preference representation.

The real LSTM stores numerical vectors rather than literal words, but this example illustrates the purpose of the gates.

## 5. Unrolling an LSTM over time

An LSTM is recurrent because the same cell is applied repeatedly. During unrolling, each time step receives:

1. The current input
2. The previous hidden state
3. The previous cell state

For a playlist application, the sequence might be:

```text
Song A -> Song B -> Song C -> Song D -> Song E
   |        |        |        |        |
  h0,c0    h1,c1    h2,c2    h3,c3    h4,c4
```

The model can use the accumulated representation from earlier songs when estimating what should come next.

### Key takeaway

Vanilla RNNs can struggle to preserve information across long sequences because gradients may shrink during backpropagation through time. LSTMs address this with a dedicated cell state and gated information flow.